In [1]:
# Performance config
import os

CPU_THREADS = min(32, os.cpu_count() or 32)
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(CPU_THREADS)
os.environ["TOKENIZERS_PARALLELISM"] = "true"

REQUIRE_CUDA = True  # set False for CPU-only notebooks

print(f"CPU threads set to: {CPU_THREADS}")

try:
    import torch
except Exception as e:
    torch = None
    if REQUIRE_CUDA:
        raise RuntimeError("CUDA required but torch is not available.") from e

if torch is not None:
    torch.set_num_threads(CPU_THREADS)
    torch.set_num_interop_threads(min(4, CPU_THREADS))
    if REQUIRE_CUDA and not torch.cuda.is_available():
        raise RuntimeError("CUDA required but not available.")
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        print("CUDA device:", torch.cuda.get_device_name(0))
    else:
        print("CUDA not available; running on CPU.")


CPU threads set to: 12
CUDA device: NVIDIA GeForce RTX 5070 Ti


# ResNet50 image-only baseline (frozen + logistic regression)

Kvasir-VQA x1: extract ResNet50 (ImageNet) features per image, train a linear classifier on top-K answers.

In [2]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report



In [3]:
# Paths & config

def find_kvasir_x1_root() -> Path:
    import os
    env_root = os.environ.get("KVASIR_VQA_X1_ROOT")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if (p / "0_dataset_prep").exists():
            return p
        raise RuntimeError(f"KVASIR_VQA_X1_ROOT set but missing 0_dataset_prep: {p}")

    if "__file__" in globals():
        p = Path(__file__).resolve()
        root = p.parents[2]
        if root.name == "Kvasir_VQA_x1" and (root / "0_dataset_prep").exists():
            return root

    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if p.name == "Kvasir_VQA_x1" and (p / "0_dataset_prep").exists():
            return p

    raise RuntimeError(
        "Could not locate Kvasir_VQA_x1 dataset root. \n"
        "Run this notebook from within the Kvasir_VQA_x1 folder, \n"
        "or set KVASIR_VQA_X1_ROOT."
    )

DATA_ROOT = find_kvasir_x1_root()
META_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "metadata_enriched.csv"
OUT_DIR = DATA_ROOT / "2_modeling" / "05_resnet50_image_only" / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 128
NUM_WORKERS = 8
TOP_K = None  # use all answers
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Data root:", DATA_ROOT)
print("Metadata:", META_CSV)
print("Out dir:", OUT_DIR)
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True



Data root: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1
Metadata: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/0_dataset_prep/out/metadata/metadata_enriched.csv
Out dir: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/05_resnet50_image_only/out
Device: cuda


In [4]:
# Load metadata
meta = pd.read_csv(META_CSV)

# Resolve image paths relative to dataset root if needed
images_base = DATA_ROOT / "0_dataset_prep"
meta["image_path"] = meta["image_path"].apply(
    lambda p: str((images_base / p).resolve()) if not Path(p).is_absolute() else p
)

meta["answer_norm"] = meta["answer"].fillna("").astype(str).str.lower().str.strip()

if "split" not in meta.columns:
    raise RuntimeError("Missing 'split' column. Run dataset prep split step first.")

train_df = meta[meta["split"] == "train"].reset_index(drop=True)
val_df = meta[meta["split"] == "validation"].reset_index(drop=True)
test_df = meta[meta["split"] == "test"].reset_index(drop=True)

print({"train": len(train_df), "val": len(val_df), "test": len(test_df)})


{'train': 143594, 'val': 0, 'test': 15955}


In [5]:
# Use all answers (no top-K filtering)
unique_answers = meta["answer_norm"].nunique()
print("Unique answers:", unique_answers)
train_k = train_df
val_k = val_df
test_k = test_df
print({"train_k": len(train_k), "val_k": len(val_k), "test_k": len(test_k)})



Unique answers: 79390
{'train_k': 143594, 'val_k': 0, 'test_k': 15955}


In [6]:
# Image embedding extraction
class ImageDS(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = row["img_id"]
        img = Image.open(row["image_path"]).convert("RGB")
        return img_id, img


def collate_fn(batch):
    ids = [b[0] for b in batch]
    imgs = [b[1] for b in batch]
    pixel_values = torch.stack([preprocess(img) for img in imgs])
    return ids, pixel_values



In [7]:
weights = ResNet50_Weights.IMAGENET1K_V2
resnet = models.resnet50(weights=weights)
resnet.fc = torch.nn.Identity()
resnet = resnet.to(DEVICE)
resnet.eval()

preprocess = weights.transforms()

# Compute or load cached embeddings
EMB_PATH = OUT_DIR / "image_embeddings_resnet50.npz"

# Unique images from filtered dataset
unique_imgs = pd.concat([train_k, val_k, test_k])[ ["img_id", "image_path"] ].drop_duplicates()

if EMB_PATH.exists():
    data = np.load(EMB_PATH, allow_pickle=True)
    emb_map = {k: v for k, v in data.items()}
    print("Loaded embeddings:", len(emb_map))
else:
    ds = ImageDS(unique_imgs)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn)
    emb_map = {}
    with torch.no_grad():
        for ids, pixels in tqdm(dl, desc="ResNet50 embeds"):
            pixels = pixels.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE.type=="cuda"):
                feats = resnet(pixels)
            feats = feats.detach().cpu().numpy()
            for i, f in zip(ids, feats):
                emb_map[i] = f
    np.savez_compressed(EMB_PATH, **emb_map)
    print("Saved embeddings to", EMB_PATH)



Loaded embeddings: 6449


In [8]:
# Build feature matrices

def build_X(df):
    return np.stack([emb_map[i] for i in df["img_id"].tolist()])

X_train = build_X(train_k)
X_val = build_X(val_k) if len(val_k) else None
X_test = build_X(test_k) if len(test_k) else None

y_train = train_k["answer_norm"].values
y_val = val_k["answer_norm"].values if len(val_k) else None
y_test = test_k["answer_norm"].values if len(test_k) else None

print("Shapes:", {"X_train": X_train.shape, "X_val": getattr(X_val, 'shape', None), "X_test": getattr(X_test, 'shape', None)})


Shapes: {'X_train': (143594, 2048), 'X_val': None, 'X_test': (15955, 2048)}


In [9]:
# Train classifier (GPU-friendly)

# Normalize features using train stats
x_mean = X_train.mean(axis=0, keepdims=True)
x_std = X_train.std(axis=0, keepdims=True) + 1e-6

X_train_n = ((X_train - x_mean) / x_std).astype(np.float32)
X_val_n = ((X_val - x_mean) / x_std).astype(np.float32) if X_val is not None else None
X_test_n = ((X_test - x_mean) / x_std).astype(np.float32) if X_test is not None else None

# Label encoding (train-only vocab)
label_list = sorted(pd.unique(y_train))
label_to_id = {lbl: i for i, lbl in enumerate(label_list)}
label_list = np.array(label_list, dtype=object)


def encode_labels(y):
    return np.array([label_to_id.get(v, -1) for v in y], dtype=np.int64)


y_train_id = encode_labels(y_train)
y_val_id = encode_labels(y_val) if y_val is not None else None
y_test_id = encode_labels(y_test) if y_test is not None else None

num_classes = len(label_list)
input_dim = X_train_n.shape[1]

# Class weights for imbalance
counts = np.bincount(y_train_id, minlength=num_classes)
weights = counts.sum() / (num_classes * np.maximum(counts, 1))
weights = torch.tensor(weights, dtype=torch.float32, device=DEVICE)

# Torch datasets/loaders
CLS_BATCH_SIZE = 1024
EPOCHS = 5
LR = 1e-3

train_ds = torch.utils.data.TensorDataset(
    torch.from_numpy(X_train_n),
    torch.from_numpy(y_train_id),
)
train_dl = torch.utils.data.DataLoader(
    train_ds,
    batch_size=CLS_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
)

model = torch.nn.Linear(input_dim, num_classes).to(DEVICE)
criterion = torch.nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    for xb, yb in tqdm(train_dl, desc=f"Train epoch {epoch}"):
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        running += loss.item() * xb.size(0)
    print(f"Epoch {epoch} loss: {running / len(train_ds):.4f}")


@torch.no_grad()
def predict_ids(X):
    model.eval()
    preds = []
    bs = CLS_BATCH_SIZE
    for start in range(0, len(X), bs):
        xb = torch.from_numpy(X[start:start + bs]).to(DEVICE, non_blocking=True)
        logits = model(xb)
        preds.append(torch.argmax(logits, dim=1).cpu().numpy())
    return np.concatenate(preds, axis=0)


def eval_split(X, y_true, split_name):
    y_true_id = encode_labels(y_true)
    mask = y_true_id >= 0
    if not np.any(mask):
        print(f"{split_name}: no labels present in training vocab")
        return
    X = X[mask]
    y_true_id = y_true_id[mask]

    y_pred_id = predict_ids(X)
    y_pred = label_list[y_pred_id]
    y_true_lbl = label_list[y_true_id]

    metrics = {
        "accuracy": float(accuracy_score(y_true_lbl, y_pred)),
        "macro_f1": float(f1_score(y_true_lbl, y_pred, average="macro")),
    }
    report = classification_report(y_true_lbl, y_pred, output_dict=True, zero_division=0)
    pred_df = pd.DataFrame({"y_true": y_true_lbl, "y_pred": y_pred})
    pred_df.to_csv(OUT_DIR / f"pred_{split_name}.csv", index=False)
    with open(OUT_DIR / f"metrics_{split_name}.json", "w") as f:
        json.dump({"metrics": metrics, "report": report}, f, indent=2)
    print(split_name, metrics)


eval_split(X_train_n, y_train, "train")
if X_val_n is not None:
    eval_split(X_val_n, y_val, "val")
if X_test_n is not None:
    eval_split(X_test_n, y_test, "test")


/home/aristotle/anaconda3/envs/vqa-rag/lib/python3.11/site-packages/numpy/core/_methods.py:187: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)


Train epoch 1:   0%|          | 0/141 [00:00<?, ?it/s]

Epoch 1 loss: 12.1746


Train epoch 2:   0%|          | 0/141 [00:00<?, ?it/s]

Epoch 2 loss: 10.3448


...

Epoch 4 loss: 11.1722


Train epoch 5:   0%|          | 0/141 [00:00<?, ?it/s]

Epoch 5 loss: 11.1230


/home/aristotle/anaconda3/envs/vqa-rag/lib/python3.11/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_true = type_of_target(y_true, input_name="y_true")
/home/aristotle/anaconda3/envs/vqa-rag/lib/python3.11/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_true = type_of_target(y_true, input_name="y_true")
/home/aristotle/anaconda3/envs/vqa-rag/lib/python3.11/site-packages/sklearn/utils/multiclass.py:79: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  ys_types = set(type_of_target(x) for x in ys)
/home/aristotle/anaconda3/envs/vqa-rag/lib/python3.11/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_true = type_of_target(y_true, input_name="y_true")
/home

train {'accuracy': 0.03495967798097414, 'macro_f1': 0.003507225807515823}
test {'accuracy': 0.002404897245299519, 'macro_f1': 7.64176318695254e-05}
